# Getting started with DuckDB in JupyterLab

This notebook checks your setup and shows the three ways we'll query the course database:

1. **SQL cells** with the `%%sql` magic (JupySQL)
2. **Python** with the `duckdb` package, returning pandas DataFrames
3. **Mixing both**, saving SQL results into Python for charts and machine learning

Run each cell with **Shift + Enter**.

## 1. Point to the course database

Change `DB_PATH` if you saved `health_analytics.duckdb` somewhere else. On Windows you can use forward slashes, e.g. `C:/Users/you/HealthAnalytics/data/health_analytics.duckdb`.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DB_PATH = Path("../data/health_analytics.duckdb")   # <- edit if needed
print("Found database:", DB_PATH.exists(), DB_PATH.resolve())

# read_only=True lets DBeaver and JupyterLab use the file at the same time
con = duckdb.connect(str(DB_PATH), read_only=True)

## 2. Turn on SQL cells

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 20
%sql con

In [ ]:
%%sql
-- Every table in the database, with its description
SELECT table_name, estimated_size AS approx_rows, comment
FROM duckdb_tables()
ORDER BY table_name

In [ ]:
%%sql
-- Claims by setting and year
SELECT claim_setting,
       YEAR(service_from_date) AS service_year,
       COUNT(*)                AS claims,
       ROUND(SUM(total_paid))  AS paid
FROM medical_claims
WHERE claim_status IN ('Paid', 'Partially Paid')
GROUP BY ALL
ORDER BY claim_setting, service_year

## 3. Save a SQL result into Python

`<<` stores the result of a `%%sql` cell in a Python variable. Then `.DataFrame()` turns it into a pandas DataFrame.

In [ ]:
%%sql enrollment <<
SELECT coverage_month, payer_type, COUNT(*) AS members
FROM coverage_member_months
GROUP BY ALL
ORDER BY coverage_month

In [ ]:
df = enrollment.DataFrame()
pivot = df.pivot(index="coverage_month", columns="payer_type", values="members")
ax = pivot.plot(figsize=(10, 5), title="Members by payer type and month")
ax.set_ylabel("Members");

## 4. Or skip the magic and use Python directly

`con.sql(...).df()` runs a query and returns a pandas DataFrame. This is handy inside functions and for ML pipelines.

In [ ]:
top_dx = con.sql("""
    SELECT c.dx_1, r.description, COUNT(*) AS ed_visits
    FROM medical_claims c
    LEFT JOIN ref_icd10 r ON r.icd10_code = c.dx_1
    WHERE c.claim_setting = 'Emergency' AND c.claim_type = 'Institutional'
    GROUP BY ALL
    ORDER BY ed_visits DESC
    LIMIT 10
""").df()
top_dx

## 5. Check the machine-learning libraries

TensorFlow is only needed for the neural network labs, so it's fine if it isn't installed yet.

In [ ]:
import sklearn, matplotlib
print("scikit-learn", sklearn.__version__)
print("matplotlib", matplotlib.__version__)
try:
    import tensorflow as tf
    print("tensorflow", tf.__version__)
except ImportError:
    print("tensorflow not installed yet (needed for neural network labs)")

## 6. Close the connection when you're done

Closing releases the file so DBeaver (or another notebook) can open it without warnings.

In [ ]:
con.close()